# Quantum Kitchen Sinks — an introductory notebook

This notebook goal is to introduce the main idea of the paper and to show, on the artificial `picture_frames` dataset, why QKS is useful and how the same idea can be ported to a photonic setting.

Outline:

1. review the idea of random kitchen sinks
2. place the paper in its quantum machine learning context
3. show the usefulness of gate-based QKS against the Linear Baseline Rule
4. show the photonic port on the same artificial dataset

## 1. What are random kitchen sinks?

**Random kitchen sinks** are a classical technique for approximating nonlinear models.

The core idea is simple:

- start from an input vector $u$
- apply a large number of fixed random transformations
- obtain a new feature space in which a **linear model** becomes much more expressive

In the classical setting, one often uses features such as

$$
\phi_j(u) = \cos(\omega_j^T u + b_j),
$$

with $\omega_j$ and $b_j$ drawn once and then frozen.

The important message is this: **we do not try to learn the nonlinear transformation itself**. We sample it at random, keep it fixed, and only train the final linear layer.

Quantum Kitchen Sinks follows exactly the same logic, but replaces the classical nonlinear feature map with a **small random quantum circuit that is never trained**.

## 2. Paper context

The paper by Wilson et al. introduces **Quantum Kitchen Sinks (QKS)** as an alternative to variational quantum models.

Instead of training a quantum circuit, the method proceeds in **episodes**:

1. encode the input $u$ into quantum angles through a random linear map $\theta_e = \Omega_e u + \beta_e$
2. execute a shallow fixed-depth circuit
3. sample a **single-shot bitstring** at the output
4. concatenate those outputs across many episodes
5. train only a final linear classifier

The central rule in the paper is the **Linear Baseline Rule (LBR or LB rule)**: any performance gain should come from the quantum nonlinearity, not from a sophisticated classical model. The reference baseline is therefore plain logistic regression on the raw inputs.

The artificial `picture_frames` dataset is ideal for introducing this idea:

- in raw 2D space, the decision boundary is not linear
- logistic regression alone stays close to chance
- QKS maps the points into a feature space where a linear model becomes sufficient

In [ ]:
from __future__ import annotations

import json
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_paper_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'lib' / 'data.py').exists() and (candidate / 'README.md').exists():
            return candidate
    raise RuntimeError('Could not locate papers/quantum_kitchen_sinks from the current working directory.')


paper_dir = find_paper_dir()
repo_root = paper_dir.parents[1]

for path in [repo_root, paper_dir]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from lib.data import load_picture_frames


def load_metrics(relative_path: str) -> dict:
    path = paper_dir / relative_path
    return json.loads(path.read_text())


def mean_std(values):
    arr = np.asarray(values, dtype=float)
    return float(arr.mean()), float(arr.std())


def summarize_results(results, group_keys):
    grouped = defaultdict(list)
    for row in results:
        key = tuple(row[k] for k in group_keys)
        grouped[key].append(float(row['test_accuracy']))
    summary = []
    for key, values in grouped.items():
        mean_acc, std_acc = mean_std(values)
        item = {name: value for name, value in zip(group_keys, key)}
        item['mean_accuracy'] = mean_acc
        item['std_accuracy'] = std_acc
        item['mean_error'] = 1.0 - mean_acc
        summary.append(item)
    return sorted(summary, key=lambda row: tuple(row[k] for k in group_keys))


GATE_RUN = 'outdir/run_20260522-121025/metrics.json'
LR_RUN = 'outdir/run_20260522-121228/metrics.json'
PHOTONIC_RUN = 'outdir/run_20260522-121424/metrics.json'

print(f'Paper directory: {paper_dir}')
print(f'Repo root:      {repo_root}')
print(f'Gate run:       {GATE_RUN}')
print(f'LR baseline:    {LR_RUN}')
print(f'Photonic run:   {PHOTONIC_RUN}')

ModuleNotFoundError: No module named 'runtime_lib'

In [2]:
X_train, y_train, X_test, y_test = load_picture_frames(n_train=1600, n_test=400, seed=0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

for ax, X, y, title in [
    (axes[0], X_train, y_train, 'Training split'),
    (axes[1], X_test, y_test, 'Test split'),
]:
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=10, alpha=0.6, label='inner frame')
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=10, alpha=0.6, label='outer frame')
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal')

axes[0].legend(loc='upper right')
plt.suptitle('The picture_frames dataset is not linearly separable in raw 2D space')
plt.show()

NameError: name 'load_picture_frames' is not defined

## 3. Why gate-based QKS is useful here

On `picture_frames`, the most pedagogical comparison is the following:

- **LR baseline**: train logistic regression directly on the two raw input coordinates
- **Gate-based QKS**: send the same input through many small random circuits, concatenate the single-shot bitstrings, and again train only logistic regression at the end

In other words, the only structural difference is the **nonlinear feature map** produced by the quantum circuit.

This notebook does not re-run all sweeps. Instead, it reads the artifacts already produced by the reproduction to display the central message of the paper.

In [3]:
lr_metrics = load_metrics(LR_RUN)
gate_metrics = load_metrics(GATE_RUN)

lr_accs = [row['test_accuracy'] for row in lr_metrics['results']]
lr_mean, lr_std = mean_std(lr_accs)

gate_summary = summarize_results(gate_metrics['results'], ['sigma', 'n_episodes'])
best_gate = max(gate_summary, key=lambda row: row['mean_accuracy'])

print('Linear baseline on raw 2D inputs')
print(f'  mean accuracy = {lr_mean:.4f}')
print(f'  mean error    = {1.0 - lr_mean:.4f}')
print()
print('Best gate-based QKS configuration on picture_frames')
print(f"  sigma      = {best_gate['sigma']}")
print(f"  episodes   = {best_gate['n_episodes']}")
print(f"  mean acc   = {best_gate['mean_accuracy']:.4f}")
print(f"  mean error = {best_gate['mean_error']:.4f}")

top_gate = sorted(gate_summary, key=lambda row: row['mean_accuracy'], reverse=True)[:8]
print()
print('Top gate-based settings')
for row in top_gate:
    print(
        f"  sigma={row['sigma']:>4}  E={row['n_episodes']:>5}  "
        f"acc={row['mean_accuracy']:.4f}  err={row['mean_error']:.4f}"
    )

NameError: name 'load_metrics' is not defined

In [ ]:
rows = gate_metrics['results']
sigmas = sorted({float(row['sigma']) for row in rows})
episodes = sorted({int(row['n_episodes']) for row in rows})
matrix = np.full((len(sigmas), len(episodes)), np.nan, dtype=float)

for i, sigma in enumerate(sigmas):
    for j, episode in enumerate(episodes):
        vals = [
            float(row['test_accuracy'])
            for row in rows
            if float(row['sigma']) == sigma and int(row['n_episodes']) == episode
        ]
        if vals:
            matrix[i, j] = np.mean(vals)

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
im = ax.imshow(matrix, aspect='auto', origin='lower', cmap='viridis', vmin=0.45, vmax=1.0)
ax.set_xticks(range(len(episodes)), [str(e) for e in episodes])
ax.set_yticks(range(len(sigmas)), [str(s) for s in sigmas])
ax.set_xlabel('Number of episodes E')
ax.set_ylabel('Sigma')
ax.set_title('Gate-based QKS on picture_frames: mean test accuracy')

for i in range(len(sigmas)):
    for j in range(len(episodes)):
        value = matrix[i, j]
        ax.text(j, i, f'{value:.2f}', ha='center', va='center', color='white', fontsize=8)

plt.colorbar(im, ax=ax, label='Mean test accuracy')
plt.show()

### How to read this figure

The linear baseline stays close to 50%, which is exactly what we expect: there is no simple affine boundary separating the two square frames.

Gate-based QKS, by contrast, adds a large collection of nonlinear features produced by small random circuits. Once those features are available, **the same logistic regression model** becomes enough to reach very high accuracy.

This is the most compact demonstration of the paper's idea: a small fixed quantum circuit can already act as a useful nonlinear feature extractor without any variational training.

## 4. Photonic port on the artificial dataset

The same recipe can be transported to a photonic setting. In this project, that is done with **MerLin**:

- a random interferometric layer plays the role of the fixed quantum part
- the classical input drives an `angle_encoding`
- one photonic output is sampled at each episode
- those outputs are concatenated and fed to a final logistic regression model

So the conceptual question is not 'is this the same hardware?' but rather: **does the QKS recipe survive a change of quantum platform?**

The `picture_frames` dataset acts as a feasibility test here. If the photonic port does not work on this simple case, there is little reason to expect it to be convincing on harder datasets.

In [ ]:
photonic_metrics = load_metrics(PHOTONIC_RUN)
photonic_accs = [row['test_accuracy'] for row in photonic_metrics['results']]
photonic_mean, photonic_std = mean_std(photonic_accs)

comparison = [
    ('Linear baseline', lr_mean, lr_std),
    ('Gate QKS (best)', best_gate['mean_accuracy'], best_gate['std_accuracy']),
    ('Photonic QKS', photonic_mean, photonic_std),
]

for label, mean_acc, std_acc in comparison:
    print(f'{label:18s}  accuracy={mean_acc:.4f}  error={1.0 - mean_acc:.4f}  std={std_acc:.4f}')

labels = [item[0] for item in comparison]
means = [item[1] for item in comparison]
stds = [item[2] for item in comparison]

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
ax.bar(labels, means, yerr=stds, capsize=6, color=['#888888', '#2a6fbb', '#c97a1f'])
ax.set_ylim(0.4, 1.05)
ax.set_ylabel('Test accuracy')
ax.set_title('On picture_frames, the photonic port preserves the central QKS idea')
for i, mean_acc in enumerate(means):
    ax.text(i, mean_acc + 0.015, f'{mean_acc:.3f}', ha='center')
plt.xticks(rotation=10)
plt.show()

## 5. Takeaways

This notebook illustrates three simple ideas:

- **random kitchen sinks** enrich the input with many fixed random features, then let a linear model do the rest
- the QKS paper replaces that classical feature map with a **quantum** feature map built episode by episode from small random circuits
- on `picture_frames`, that idea is clearly useful in the gate-based setting against the linear baseline, and the **photonic port** preserves that usefulness on the same artificial task

In other words, the goal of QKS is not to claim that every problem becomes easy with a quantum circuit. The point is more modest and more interesting: a **small untrained quantum device** can already serve as a meaningful nonlinear feature extractor.

If you want to extend this notebook, the two most natural next steps are:

1. add an MNIST section to show why the comparison becomes much more delicate in high dimension
2. add a cell that explicitly re-runs one small gate-based config and one small photonic config from the project's `configs/*.json` files